In [ ]:
%%bigquery
-- I'm creating a new dataset to keep my work for this challenge organized.
CREATE SCHEMA IF NOT EXISTS weather_dataset
OPTIONS(location="US");


Query is running:   0%|          |

""


In [ ]:
%%bigquery
-- I'm telling BigQuery to load the data from the lab's GCS bucket.
-- It will overwrite the table if it already exists, skip the header row, and auto-detect the schema.
LOAD DATA OVERWRITE weather_dataset.raw_weather
FROM FILES (
  format = 'CSV',
  uris = ['gs://labs.roitraining.com/data-to-ai-workshop/weather_data.csv'],
  skip_leading_rows = 1
);


Query is running:   0%|          |

""


In [ ]:
# Part A: I'll enable the necessary APIs for my project first.
!gcloud services enable bigqueryconnection.googleapis.com --project=qwiklabs-gcp-01-ed33dcbea5fb
!gcloud services enable aiplatform.googleapis.com --project=qwiklabs-gcp-01-ed33dcbea5fb


In [ ]:
!gcloud config set project qwiklabs-gcp-01-ed33dcbea5fb

Project 'qwiklabs-gcp-01-ed33dcbea5fb' lacks an 'environment' tag. Please create or add a tag with key 'environment' and a value like 'Production', 'Development', 'Test', or 'Staging'. Add an 'environment' tag using `gcloud resource-manager tags bindings create`. See https://cloud.google.com/resource-manager/docs/creating-managing-projects#designate_project_environments_with_tags for details.
Updated property [core/project].


In [ ]:
!bq mk --connection --location=US --project_id=qwiklabs-gcp-01-ed33dcbea5fb --connection_type=CLOUD_RESOURCE gemini_conn

Connection 820324039680.us.gemini_conn successfully created


In [ ]:
%%bigquery

-- I'm creating a model that uses my connection to talk to the 'gemini-2.5-flash' endpoint.

CREATE OR REPLACE MODEL weather_dataset.gemini_model

REMOTE WITH CONNECTION DEFAULT

OPTIONS(ENDPOINT = 'gemini-2.5-flash');

Query is running:   0%|          |

""


In [ ]:
%%bigquery
-- I'm creating a final table called 'weather_with_reports' to store the results.
CREATE OR REPLACE TABLE weather_dataset.weather_with_reports AS
SELECT
  *,
  -- I'm extracting only the generated text from the model's output.
  ml_generate_text_result['content'] AS weather_report
FROM
  ML.GENERATE_TEXT(
    MODEL `weather_dataset.gemini_model`,
    (
      -- For each row of my raw data, I'm building a custom prompt for Gemini.
      SELECT
        *,
        CONCAT(
          'Based on a temperature of ', temperature_f,
          ' degrees Fahrenheit, wind speed of ', wind_speed_mph,
          ' mph, and humidity of ', humidity_percent,
          '%, write a short, professional weather warning or report.'
        ) AS prompt
      FROM
        `weather_dataset.raw_weather`
    ),
    -- I'm giving Gemini some final instructions on how to behave.
    STRUCT(
      0.2 AS temperature, -- This makes the model more factual.
      250 AS max_output_tokens -- This limits the report length.
    )
  );


Query is running:   0%|          |

""


In [23]:
%%bigquery
-- Let's see the results of all my work!
SELECT
  temperature_f,
  wind_speed_mph,
  weather_report
FROM
  weather_dataset.weather_with_reports
LIMIT 10;


Query is running:   0%|          |

Downloading:   0%|          |

,temperature_f,wind_speed_mph,weather_report
0,91.0,4.0,None
1,70.0,3.0,None
2,23.5,9.0,None
3,68.4,3.0,None
4,59.8,5.0,None
5,67.5,8.0,None
6,33.0,16.0,None
7,60.0,10.3,None
8,44.0,35.0,None
9,76.0,0.3,None
